In [1]:
!pip install "transformers>=4.42" "trl>=0.9.4" peft accelerate datasets bitsandbytes

In [2]:
!python -m pip uninstall -y bitsandbytes
!python -m pip install -U bitsandbytes

Found existing installation: bitsandbytes 0.48.1
Uninstalling bitsandbytes-0.48.1:
  Successfully uninstalled bitsandbytes-0.48.1
  Using cached bitsandbytes-0.48.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.48.1-py3-none-manylinux_2_24_x86_64.whl (60.1 MB)


In [3]:
import os, torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # single-GPU pin
if torch.cuda.is_available():
    torch.cuda.set_device(0)

In [4]:
# !pip install -U "trl>=0.21.0" "transformers>=4.55.0" accelerate peft datasets bitsandbytes

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig

MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
TRAIN, EVAL = "train.jsonl", "eval.jsonl"

tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 2048   # tighten to ease VRAM; raise later if room

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)
model.config.use_cache = False  # required with grad checkpointing

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

ds = load_dataset("json", data_files={"train": TRAIN, "eval": EVAL})

def to_prompt_completion(ex):
    return {
        "prompt": ex["problem"] + "\n\nAnswer:",
        "completion": ex["code"]
    }

ds = ds.map(to_prompt_completion, remove_columns=ds["train"].column_names)

cfg = SFTConfig(
    output_dir="qwen-coder-sft",
    per_device_train_batch_size=1,      # keep at 1 on L4
    gradient_accumulation_steps=16,     # increase if you want bigger effective batch
    num_train_epochs=10,
    learning_rate=1e-4,
    logging_steps=20,
    eval_strategy="no",                 # turn off eval if memory is tight; or "steps"
    save_strategy="steps",
    save_steps=400,
    completion_only_loss=True,          # trains only on the completion (code)
    packing=False,                      # packing can raise peak mem; enable later if stable
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",           # bitsandbytes optimizer = lower RAM/VRAM
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    peft_config=peft_config,
    train_dataset=ds["train"],
    eval_dataset=ds.get("eval"),
)

trainer.train()
trainer.save_model("qwen-coder-sft")
tokenizer.save_pretrained("qwen-coder-sft")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.33G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

MODEL = "qwen-coder-sft"   # path to your saved fine-tuned model folder

# Load tokenizer & model (from local fine-tune output)
tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto", torch_dtype="auto")

# Build a text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype="auto",
    #device=0
)

# Test problem
prompt = (
    "You are a helpful coding assistant that will draw a diagram for given problems using code. \
    ONLY write code; don't explain a solution. \n\n"
    "Problem: A block of mass m rests on a 30° incline. Draw a free body diagram with W, N, and friction f, and draw decomposition of weight into parallel and perpendicular directions to the incline."
    "Answer:"
)

out = pipe(prompt, max_new_tokens=1500, do_sample=False)
print(out[0]["generated_text"])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create figure and axis
fig, ax = plt.subplots()

# Define angle and mass (symbolic)
theta = np.radians(30)  # Convert degrees to radians
m ='m'  # Mass symbol

# Draw inclined plane
x_plane = np.array([0, 1])
y_plane = x_plane * np.tan(theta)
ax.plot(x_plane, y_plane, label='Incline')

# Draw block at top of incline
block_x = 0.5
block_y = 0.5 * np.tan(theta)
ax.scatter(block_x, block_y, s=200, color='blue', label='Block')

# Free body diagram components
W = np.array([[block_x], [block_y]])
N = np.array([[block_x - 0.1], [block_y + 0.1]])
f = np.array([[block_x + 0.1], [block_y]])

# Draw vectors
ax.quiver(W[0], W[1], 0, -np.sin(theta), angles='xy', scale_units='xy', scale=1, color='red', label='Weight (W)')
ax.quiver(N[0], N[1], 0, np.cos(theta), angles='xy', scale_units='xy', scale=1, color='green', label='Normal Force (N)')
ax.quiver(f[0], f[1], np.cos(theta), np.sin(theta), angles='xy', scale_units='xy', scale=1, color='orange', label='Friction (f)')

# Decomposition of weight
W_parallel = np.array([[block_x], [block_y - 0.1]])
W_perp = np.array([[block_x], [block_y + 0.1]])

# Draw decomposition vectors
ax.quiver(W_parallel[0], W_parallel[1], 0, -np.sin(theta), angles='xy', scale_units='xy', scale=1, color='purple', linestyle='--')
ax.quiver(W_perp[0], W_perp[1], 0, np.cos(theta), angles='xy', scale_units='xy', scale=1, color='purple', linestyle='--')

# Add labels and legend
ax.set_xlabel('X-axis')
ax.set_ylabel('Y-axis')
ax.legend()
plt.show()

In [ ]:
# diagram_02.py — Single free-body diagram on an incline.
import matplotlib.pyplot as plt
import numpy as np

# --- Parameters for this diagram ---
angle_deg = 15
flip = 1  # 1 for normal, -1 for left-right mirror
plane_color, box_color, weight_color, par_color, perp_color, normal_color = ('saddlebrown', 'teal', 'crimson', 'indigo', 'darkorange', 'darkgreen')
box_size = 0.45
line_w = 3

# --- Helpers ---
def draw_incline(ax, angle_deg, flip, color, linewidth):
    angle_rad = np.deg2rad(angle_deg)
    x = np.array([0, 2*np.cos(angle_rad)]) * flip
    y = np.array([0, 2*np.sin(angle_rad)])
    ax.plot(x, y, color=color, linewidth=linewidth)

def draw_box(ax, angle_deg, flip, color, box_size, linewidth):
    angle_rad = np.deg2rad(angle_deg)
    w = h = box_size
    x0, y0 = 0.8*np.cos(angle_rad), 0.8*np.sin(angle_rad)
    R = np.array([[np.cos(angle_rad), -np.sin(angle_rad)],
                  [np.sin(angle_rad),  np.cos(angle_rad)]])
    pts = np.array([[0,0],[w,0],[w,h],[0,h],[0,0]], dtype=float)
    world = (R @ pts.T).T + np.array([x0, y0])
    world[:,0] *= flip
    ax.plot(world[:,0], world[:,1], color=color, linewidth=linewidth)
    center = np.array([x0 + (w/2)*np.cos(angle_rad), y0 + (h/2)*np.sin(angle_rad)])
    center[0] *= flip
    return center

def draw_arrow(ax, start, vec, color, label=None, label_offset=(0,0)):
    ax.arrow(start[0], start[1], vec[0], vec[1],
             head_width=0.06, head_length=0.12, fc=color, ec=color, length_includes_head=True)
    if label:
        ax.text(start[0] + vec[0]*1.05 + label_offset[0],
                start[1] + vec[1]*1.05 + label_offset[1],
                label, fontsize=11, color=color)

# --- Build the diagram ---
fig, ax = plt.subplots(figsize=(5,5))

draw_incline(ax, angle_deg, flip, plane_color, line_w)
center = draw_box(ax, angle_deg, flip, box_color, box_size, line_w)

angle_rad = np.deg2rad(angle_deg)

# directions (mirror x if flipped)
normal_dir  = np.array([-np.sin(angle_rad), np.cos(angle_rad)], dtype=float)
tangent_dir = np.array([ np.cos(angle_rad), np.sin(angle_rad)], dtype=float)
normal_dir[0]  *= flip
tangent_dir[0] *= flip

# Forces
W = np.array([0.0, -0.8])                         # weight (downward)
W_par  = -tangent_dir * np.linalg.norm(W) * np.sin(angle_rad)  # along plane
W_perp = -normal_dir  * np.linalg.norm(W) * np.cos(angle_rad)  # into plane
N      =  normal_dir * np.linalg.norm(W) * np.cos(angle_rad)   # normal

# Draw
draw_arrow(ax, center, W,        weight_color, 'mg',       (0,-0.12))
draw_arrow(ax, center, W_par,    par_color,    'mg sinθ',  (-0.2*flip, 0))
draw_arrow(ax, center, W_perp,   perp_color,   'mg cosθ',  (0,-0.1))
draw_arrow(ax, center, N,        normal_color, 'N',        (0, 0))

# projection helper lines
W_tip = center + W
par_tip = center + W_par
perp_tip = center + W_perp
ax.plot([W_tip[0], par_tip[0]], [W_tip[1], par_tip[1]], linestyle='--', color=weight_color, linewidth=1)
ax.plot([W_tip[0], perp_tip[0]],[W_tip[1], perp_tip[1]], linestyle='--', color=weight_color, linewidth=1)

# Visuals
ax.set_aspect('equal')
ax.axis('off')
ax.set_xlim(-2.0, 2.2)
ax.set_ylim(-1.4, 2.0)
title_text = "Diagram 02: θ=" + str(angle_deg) + "°" + (" (flipped)" if flip==-1 else "")
ax.set_title(title_text)
plt.tight_layout()

# Save image and show
out_png = "diagram_02.png"
plt.savefig(out_png, dpi=200, bbox_inches='tight')
plt.show()
print("Saved:", out_png)
